# BANA 4373 / ECON 4370 — Lecture 2 (First API Call)
## U.S. Census API (ACS) — Median Household Income & Population

**Goal:** Make your first API request, understand the JSON response, and load it into a clean `pandas` DataFrame.

We’ll use the **American Community Survey (ACS) 5-year** data and pull:
- `B19013_001E` = **Median household income (estimate)**
- `B01003_001E` = **Total population (estimate)**

We’ll start with **Texas (state FIPS = 48)** and then extend to **Texas counties**.


## 0. Setup
Run the cell below to import packages.

> If you get `ModuleNotFoundError: requests`, run:  
> `pip install requests` (or `conda install requests`) in your environment.


In [1]:
import requests
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 1. What does an API return?
An API typically returns **structured data** (often JSON). In many cases, it’s just a **table in disguise**.

### Try in a browser (no code)
Copy/paste this URL into your browser:

```
https://api.census.gov/data/2022/acs/acs5?get=NAME,B19013_001E,B01003_001E&for=state:48
```

You should see:
- First row: column names  
- Next row(s): data values


## 2. Make the same request in Python (Texas only)
Now we will request the same data programmatically.


In [3]:
base = "https://api.census.gov/data/2022/acs/acs5"
params = {
    "get": "NAME,B19013_001E,B01003_001E",
    "for": "state:48"  # Texas
}

r = requests.get(base, params=params, timeout=30)
r.raise_for_status()

data = r.json()
data[:2]  # header row + first data row


[['NAME', 'B19013_001E', 'B01003_001E', 'state'],
 ['Texas', '73035', '29243342', '48']]

## 3. Convert JSON to a DataFrame
The Census API returns a list of rows:
- `data[0]` = header
- `data[1:]` = records

We’ll convert numeric fields to numbers.


In [ ]:
header = data[0]
rows = data[1:]

df_tx = pd.DataFrame(rows, columns=header)

# Convert numeric columns (they arrive as strings)
for col in ["B19013_001E", "B01003_001E"]:
    df_tx[col] = pd.to_numeric(df_tx[col], errors="coerce")

df_tx


### Interpret the results (quick check)
- Is the income in **dollars**?
- Is population a **count**?
- Do you see any missing values (`NaN`)? Why might that happen?


## 4. Expand to Texas counties (all counties)
Now we request the same variables for **every county in Texas**.

Browser URL (optional to preview):
```
https://api.census.gov/data/2022/acs/acs5?get=NAME,B19013_001E,B01003_001E&for=county:*&in=state:48
```


In [ ]:
params_counties = {
    "get": "NAME,B19013_001E,B01003_001E",
    "for": "county:*",
    "in": "state:48"
}

r = requests.get(base, params=params_counties, timeout=30)
r.raise_for_status()
data_c = r.json()

df_counties = pd.DataFrame(data_c[1:], columns=data_c[0])
for col in ["B19013_001E", "B01003_001E"]:
    df_counties[col] = pd.to_numeric(df_counties[col], errors="coerce")

df_counties.head()


## 5. Basic sanity checks (cleaning mindset)
Before analysis, always check:
- duplicates
- missingness
- ranges / outliers
- units and definitions (metadata)


In [ ]:
df_counties.shape, df_counties.duplicated().sum(), df_counties.isna().sum()


In [ ]:
df_counties[["B19013_001E","B01003_001E"]].describe()


## 6. Quick analysis: top counties by median household income
This is just a demo (we will do better visualizations later).


In [ ]:
top10 = df_counties.sort_values("B19013_001E", ascending=False).head(10)
top10[["NAME","B19013_001E","B01003_001E","state","county"]]


## 7. (Optional) Change geography or year
- Change `state:48` to another state FIPS (e.g., 06 = California, 12 = Florida)
- Change the year in the base URL (e.g., 2021, 2023 if available)

### State FIPS examples
- Texas = 48
- California = 06
- Florida = 12
- New York = 36


In [ ]:
# Try another state here by changing the FIPS code.
# Example: California
params_ca = {
    "get": "NAME,B19013_001E,B01003_001E",
    "for": "state:06"
}

data_ca = requests.get(base, params=params_ca, timeout=30).json()
df_ca = pd.DataFrame(data_ca[1:], columns=data_ca[0])
for col in ["B19013_001E", "B01003_001E"]:
    df_ca[col] = pd.to_numeric(df_ca[col], errors="coerce")

df_ca


## 8. Short reflection (write 3–5 sentences)
1. What surprised you about the API output?
2. What steps did we take to make the data analysis-ready?
3. Why is this process more reproducible than downloading an Excel file?
